![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 41 -- Lab 1: Faulty or Fine

A factory checks every product that leaves its production line, but a full quality check is slow and costly, so only a few products were ever checked by hand. You write the program that gives every new product a probability of being defective from the line's sensor readings. Your file is scored with ROC-AUC: how well your numbers put the defective products above the fine ones.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Sensor and process readings of 15,000 products of one production line. 150 products were checked by hand (`defect`: 1 defective, 0 fine), 12,850 were not, and 2,000 are the test, whose answers only the scorer holds. The setup cell leaves in `DATA_DIR`: `train.csv` (`id`, the 32 readings, `line`, `defect`), `unlabeled.csv` (the same columns without `defect`), `test.csv` (the same) and `sample_submission.csv` (the format of the file you hand in).

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="uns-02", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("libraries ready")


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/faulty-or-fine")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The task

**The story.** On a production line, every product is measured by sensors while it is made. A full quality
check by hand is slow and costly, so only a small part of the products was checked. You must estimate, for every
test product, whether it is **defective**.

**The data.** Every product has an `id`, 32 readings and `line`, the production line it was made on:

- *physical sensor data:* `temperature` (degrees C), `pressure` (bar), `vibration` (mm/s), `voltage` (V), `torque`
  (Nm), `humidity` (%), `rpm` (rotational speed);
- *maintenance data:* `tool_wear` (degree of wear of the tool), `hours_since_service` (hours since the last
  maintenance);
- *derived features:* `temp_x_pressure` (temperature times pressure, scaled), `vibration_per_torque`, `wear_x_hours`
  (scaled), `voltage_per_rpm` (scaled), `humidity_x_temp` (scaled);
- *sensor measurements:* `sensor_01` to `sensor_18`, other sensor values from the production line;
- `line`: `Line_A` to `Line_E`.

`train.csv` has 150 products checked by hand, with `defect` (1 defective, 0 fine). `unlabeled.csv` has 12,850
products with the same readings **but no answer**. `test.csv` has the products you answer for. The data is
entirely synthetic, so there is no point in looking for it elsewhere. The organisers add: *the training data has
two parts, a labelled one and an unlabelled one; it is worth using the unlabelled data as well.*

**The file you hand in** has one row per test product: its `id`, and `defect`, a **probability between 0 and 1**
(1 certainly defective, 0 certainly fine).

**The score** is ROC-AUC on the test products, computed by the scorer cell in Part 2.

**The rules.** Any library; no outside data. Your notebook must run from start to end.

**The starter.** The organisers' starter (next cell) fits a logistic regression on all the readings of the 150
labelled products. The score of its file is the **baseline**; your points count from there upward.

**Today you work alone.** Nobody helps with the code and AI assistants are off. This morning's slides are yours
to use.


---
# The starter solution

Every olympiad task comes with a starter solution: a short notebook that reads the files and hands in a valid file. Its score is the **baseline**, and your points count up from it. This is the starter of this task, as the organisers gave it. Read it before you begin; you may reuse any of its lines.

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler

train_labeled = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_unlabeled = pd.read_csv(os.path.join(DATA_DIR, "unlabeled.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# Preparation: we convert the line column to numeric
le = LabelEncoder()
all_lines = pd.concat([train_labeled['line'], train_unlabeled['line'], test_df['line']])
le.fit(all_lines)

# Features and target variable from the labeled data
X_train = train_labeled.drop(columns=['id', 'defect']).copy()
X_train['line'] = le.transform(X_train['line'])
y_train = train_labeled['defect']

# Test data
X_test = test_df.drop(columns=['id']).copy()
X_test['line'] = le.transform(X_test['line'])

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Training the logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Prediction on the test data (probability values)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

submission = pd.DataFrame({'id': test_df['id'], 'defect': y_pred_proba})
submission.to_csv('submission.csv', index=False)
print(f"Submission file saved: {len(submission)} rows")
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at the data before you build anything.

## Task 1: Load the files

Read the four CSV files of `DATA_DIR` into `train`, `unlabeled`, `test` and `sample`. Print the shape of each.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, unlabeled=unlabeled, test=test, sample=sample)


## Task 2: Look at the labelled products

Store the first five rows of `train` in `first_rows` and its number of rows in `n_rows`. Print `first_rows`, then how many products of `train` are defective and how many are fine.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_rows, size=n_rows)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 7. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears.

The scorer of this task. Run the cell; you will use `score` all afternoon.

In [ ]:
from sklearn.metrics import roc_auc_score


def score(answers, numbers):
    """The score of a file: ROC-AUC of its numbers against the answers (1 defective, 0 fine)."""
    return float(roc_auc_score(answers, numbers))


print(score([0, 0, 1, 1], [0.2, 0.6, 0.3, 0.9]))


## Task 3: What the scorer gives

The scorer cell above prints 0.75 for four products. Which sentence says what `score` gives?

- **a** The share of products whose number, rounded to 0 or 1, is equal to the answer: 1.0 when every product is right, 0.5 when half of them are.
- **b** The share of (defective, fine) pairs where the defective product has the higher number, a tie counting one half: 1.0 for a perfect order, 0.5 for a constant file.
- **c** One minus the mean distance between a product's number and its answer: 1.0 when every number equals the answer, 0.5 for a constant file.

Store the letter in `formula`.

In [ ]:
# Your code here


## Task 4: Two answers without a model

What would two files that need no model score? `constant` gives every product the number 0.5; `temperature` uses the product's own `temperature` reading as its number. Score both on the products of `train` against their `defect` with `score`, and store the two numbers in the dictionary `trivial` under these two names.

In [ ]:
# Your code here


## Task 5: What the metric rewards

File A holds a model's probabilities for the test products. File B holds the same model's answers rounded to 0 or 1. File C is file A with every number divided by 2. What do they score?

- **a** B scores at least as much as A, because rounding removes the doubt, and C scores half of A.
- **b** C scores exactly as much as A, because only the order counts; B mostly scores less, since its rows tie.
- **c** All three score the same, because the three files give every product the same final answer.

Store the letter in `rewards`.

In [ ]:
# Your code here


## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, `id_col`, the name of its id column, and `pred_col`, the name of the column of your numbers. Read all three from what you loaded; do not type them.

In [ ]:
# Your code here


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
# Your code here


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- A first file

The simplest pipeline that ends in a file comes first: aim to hand it in within the first hour, then make it better.

## Task 8: Hold out validation products

Split `train` into a part `tr` and a validation part `va` of 30 products, with `random_state=0`. Keep the ids of the two parts in `train_ids` and `val_ids`. Print the number of products in each part.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=train_ids, val_ids=val_ids)


## Task 9: The numbers the model reads

Build `X_tr`, the table of numbers your model reads for the products of `tr` (one row per product), and `y_tr`, their answers. Build `X_va` and `y_va` the same way for `va`. Print the shapes.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(X_tr, y_tr)


## Task 10: A first model

Train a first model in `model` on `X_tr` and `y_tr`. It must give a probability per product. Print what it gives the first three products of `X_va`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, X_va)


## Task 11: Score the validation products

Store in `val_score` the `score` of your model's probabilities of a defect for the products of `va` against `y_va`: one number between 0 and 1. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="roc_auc")


## Task 12: Write the submission file

Write `submission.csv` in the format of `sample`: for every product of `test`, its `id` and its probability of a defect. Leave the index out. Print its shape and its first rows.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 4 -- Debugging drill

A classmate tried to get the probability of a defect of the first validation product alone. The cell stops with an error. Run the first cell: it starts the drill's clock. Then repair the classmate's cell until it prints `drill_proba`, and run the check. The clock stops at the first check that passes. Read the error message before you change anything.

In [ ]:
ck.drill_start("bug-shp-1")


## Task 13: Repair the classmate's cell

Change as little as you can: the product and the model stay.

In [ ]:
first_product = X_va[0]
drill_proba = model.predict_proba(first_product)
print(drill_proba)


In [ ]:
ck.drill_check("bug-shp-1", drill_proba)


---
# Part 5 -- Challenge: your best file

Your first file is on the board. Now make it better, one change at a time. A change worth keeping comes from something you found in the data or in the rules, and it raises the score of the validation products.

## Task 14: Look again

Look again at the data, all of it, for something your first file does not use yet. Print what you find.

In [ ]:
# Your code here


## Task 15: Try a better way

Make at least one change to your pipeline. Score it on the products of `va` with `score`, and print it next to `val_score`.

In [ ]:
# Your code here


## Task 16: Hand in your best file

Write `submission.csv` again with your best way. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
## Task 17: Reflect

Which change moved your validation score most, and how did you find it: in the data, in the rules or by trying? What did the unlabelled products tell you that the 150 labelled ones could not? If you met this task again with one hour, what would you do in the first 15 minutes?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task first**: the scorer read as code, two files without a model, what ROC-AUC rewards, and the format of the file, all before any model.
- **A first honest file**: a simple model, scored on validation products with the task's own metric, handed in early.
- **All of the data is data**: what the products have in common, found by looking and counting, and used.
- **ROC-AUC rewards an order**: probabilities, not rounded answers, in the file.
- **Debugging**: a shape error read, located and repaired against the clock.

---

Made By **Sattam Altwaim**